<div align="center">

### RR Skillverse — Free Learning Handbook
**by Raushan Ranjan**

*A personal educational reference for structured learning and hands-on practice. Shared for learning purposes only — not a commercial product or paid service.*

</div>

---

# Module 6 — Explainable & Responsible AI

**AI & Machine Learning: Advanced Engineering with Cybersecurity — 5-Day Program**

*Module 6 of 12 · 3 hours · Continues the RR Finance system built in Modules 1–5*

## Recap — what RR Finance already has

Module 1 built the trusted tabular foundation (a logistic-regression baseline, ROC-AUC ≈ 0.69, plus clustering and an Isolation Forest poisoning defence) and **deliberately excluded `age` from every model's features**, flagging it for a fairness audit "in a later module." Module 2 added a deeper ANN classifier and a CNN for cheque digits, and showed both models can be attacked and partially hardened. Module 3 added text understanding. Module 4 built and aligned a language model from scratch. Module 5 gave RR Finance memory (RAG) and agency (tool-using agents), and closed by showing those agents can be attacked too.

**Every module so far has answered "does it work?" and, increasingly, "can it be attacked?" Module 6 asks a third question RR Finance has been putting off since Module 1: "can we explain *why* it made this decision, and is that decision fair?"**

This matters for a very concrete reason: RR Finance's loan-default classifier rejects real applicants. Under fair-lending law in most jurisdictions (e.g. the US Equal Credit Opportunity Act / Regulation B), a rejected applicant is legally entitled to the *specific reasons* for that rejection — not "the model said no." A black-box score is not a compliant answer. That legal reality is what turns "explainability" from a nice-to-have into infrastructure.

**Module 6 finally pays off the single most-repeated forward reference in this course:** the `age` column, carried in the dataframe since Module 1, excluded from `FEATURES` every module since, "flagged for Module 6." That audit happens in Lesson 6 below.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json
import joblib

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

Path("data").mkdir(exist_ok=True)
Path("artifacts").mkdir(exist_ok=True)

print("numpy:", np.__version__, "| pandas:", pd.__version__)
print("Project folders ready: data/, artifacts/")

In [ ]:
%pip install -q shap lime fairlearn torch numpy pandas matplotlib scikit-learn joblib
print("Setup complete -- if you saw 'Requirement already satisfied' lines above, that is expected and fine.")

---
## Lesson 1 — Recap: loading what Modules 1 and 2 actually built

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Before we can explain RR Finance's decisions, we need the *actual* trained models making those decisions — not stand-ins. |
| **2. Why does it matter in finance?** | An explanation of a model nobody actually deployed is worthless; regulators and applicants care about the real decision-maker. |
| **3. Why this technique?** | Module 1 persisted its baseline pipeline to `artifacts/baseline_logreg_pipeline.joblib` — we load that real artifact directly, exactly as Module 2 did. |
| **4. What do the parameters mean?** | N/A — this is a load step, not a training step. |
| **5. What is happening mathematically?** | N/A. |
| **6. What happens if we change it?** | If the artifact is missing, every module since Module 2 has fallen back to retraining locally with the same `SEED`, `FEATURES`, and split — we do the same. |

**One honest gap to flag up front:** Module 2 trained its `DefaultANN` in-notebook but — unlike Module 1 — never persisted it to `artifacts/`. So there is no `module2_ann.pt` to load. We close that gap the same way Module 2 closed its own gap when Module 1's artifact was missing: **retrain it locally, using the exact architecture, feature order, split, and `SEED` from Module 2's notebook**, so the reproduction is faithful rather than approximate.

In [ ]:
DATA_PATH = Path("data/rr_finance_module1_dataset_enriched.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}.\n"
        "Place rr_finance_module1_dataset_enriched.csv (Module 1's enriched output, which "
        "still carries the 'age' column) inside a 'data' folder next to this notebook."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH.resolve())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)

In [ ]:
MODULE1_METRICS_PATH = Path("artifacts/module1_metrics.json")
BASELINE_PATH = Path("artifacts/baseline_logreg_pipeline.joblib")

if MODULE1_METRICS_PATH.exists():
    with open(MODULE1_METRICS_PATH) as f:
        module1_metrics = json.load(f)
    print("Loaded Module 1's ACTUAL saved metrics:")
    print(f"  Dataset rows:            {module1_metrics['dataset_rows']}")
    print(f"  Default rate:            {module1_metrics['default_rate']:.4f}")
    print(f"  Baseline logreg test AUC: {module1_metrics['baseline_logreg_test_auc']:.4f}")
else:
    module1_metrics = None
    print("module1_metrics.json not found -- continuing without it (standalone run).")

FEATURES = [
    "annual_income", "monthly_debt", "loan_amount", "loan_term_months",
    "credit_score", "employment_years", "account_age_months",
    "num_previous_loans", "previous_defaults", "debt_to_income", "loan_to_income",
]
# NOTE: "age" is STILL excluded from FEATURES here, exactly as every prior module --
# it is loaded separately below, purely as an audit variable for Lesson 6.

from sklearn.model_selection import train_test_split

X = df[FEATURES]
y = df["default"]
age = df["age"]

X_train, X_test, y_train, y_test, age_train, age_test = train_test_split(
    X, y, age, test_size=0.20, stratify=y, random_state=SEED
)

if BASELINE_PATH.exists():
    baseline_pipeline = joblib.load(BASELINE_PATH)
    print("\nLoaded Module 1's ACTUAL saved baseline pipeline (scaler + logistic regression).")
else:
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    baseline_pipeline = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=2000))])
    baseline_pipeline.fit(X_train, y_train)
    print("\nartifacts/baseline_logreg_pipeline.joblib not found -- retrained locally as a fallback.")

from sklearn.metrics import roc_auc_score
logreg_test_probs = baseline_pipeline.predict_proba(X_test)[:, 1]
print(f"Reproduced baseline logreg test AUC on THIS split: {roc_auc_score(y_test, logreg_test_probs):.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(SEED)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

class DefaultANN(nn.Module):
    """Reproduces Module 2's DefaultANN exactly: same layer sizes, same activation, same raw-logit output."""
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
        )
    def forward(self, x):
        return self.net(x)

model_ann = DefaultANN(n_features=X_train_t.shape[1])
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_ann.parameters(), lr=0.01)

EPOCHS = 150
model_ann.train()
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    logits = model_ann(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | training loss: {loss.item():.4f}")

model_ann.eval()
with torch.no_grad():
    ann_test_probs = torch.sigmoid(model_ann(X_test_t)).numpy().ravel()
ann_test_auc = roc_auc_score(y_test, ann_test_probs)
print(f"\nReproduced Module 2 ANN -- test ROC-AUC on THIS split: {ann_test_auc:.4f}")
print("(Module 2's own notebook reported an ANN test AUC in the same range on its identical split;")
print(" small differences from run-to-run are expected -- neural net training is not perfectly")
print(" deterministic across library/version boundaries even with a fixed SEED. This is disclosed,")
print(" not hidden, per this project's testing discipline.)")

---
## Lesson 2 — SHAP on the Module 1 logistic regression: global and individual explanations

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Which features actually drive the baseline model's default predictions, both across the whole portfolio and for one specific applicant? |
| **2. Why does it matter in finance?** | This is the difference between "the model is 69% accurate" (useless to a rejected applicant) and "your debt-to-income ratio and credit score pulled your score down" (a compliant adverse-action reason). |
| **3. Why this technique?** | **SHAP (SHapley Additive exPlanations)** is model-agnostic, grounded in cooperative game theory, and gives explanations that are consistent and additive — the contributions of every feature sum exactly to the difference between the prediction and the average prediction. That additivity is what LIME (Lesson 3) does not guarantee. |
| **4. What do the parameters mean?** | We use `shap.Explainer` with the pipeline's `predict_proba`, background data sampled from the training set (what SHAP compares each prediction against), and a `KernelExplainer`-style model-agnostic estimation since logistic regression here sits inside a `Pipeline` with a scaler. |
| **5. What is happening mathematically?** | See the Math & Algorithm toggle on the published handbook page for the Shapley value formula. |
| **6. What happens if we change it?** | A smaller background sample runs faster but with noisier SHAP estimates; we use a modest sample size that is stable enough for a teaching notebook. |

In [ ]:
import shap

# Model-agnostic explainer over the full pipeline (scaler + logistic regression),
# so SHAP values are expressed in the ORIGINAL feature units -- what an applicant actually sees on a statement.
background = shap.sample(X_train, 100, random_state=SEED)
explainer_logreg = shap.Explainer(baseline_pipeline.predict_proba, background, seed=SEED)

# Explain a manageable slice of the test set (SHAP's model-agnostic estimator is O(features x samples))
X_explain = X_test.iloc[:80]
shap_values_logreg = explainer_logreg(X_explain)

# shap_values_logreg has one SHAP value per class; we want the "default" (class 1) output
shap_values_default = shap_values_logreg[..., 1]

plt.figure()
shap.summary_plot(shap_values_default, X_explain, show=False)
plt.title("SHAP global feature importance -- Module 1 logistic regression")
plt.tight_layout()
plt.show()

In [ ]:
# Pick ONE real rejected applicant (highest predicted default probability in this slice) to explain individually
probs_explain = baseline_pipeline.predict_proba(X_explain)[:, 1]
applicant_idx = int(np.argmax(probs_explain))
applicant_row = X_explain.iloc[applicant_idx]

print(f"Applicant under review (test-set row {applicant_idx}):")
print(applicant_row)
print(f"\nModel's predicted default probability: {probs_explain[applicant_idx]:.3f}")
print(f"Actual outcome (ground truth, for our own reference only): "
      f"{'defaulted' if y_test.iloc[X_test.index.get_loc(X_explain.index[applicant_idx])] == 1 else 'did not default'}")

plt.figure()
shap.plots.waterfall(shap_values_default[applicant_idx], show=False)
plt.tight_layout()
plt.show()

**Reading the waterfall plot:** each bar is one feature's SHAP value for this one applicant — how many probability points that feature pushed the prediction up (red, toward "default") or down (blue, toward "no default") relative to the average applicant. The bars sum exactly to `base value + Σ(SHAP values) = this applicant's predicted probability`. This is what a real adverse-action explanation is built from: "your application was declined primarily because of X and Y" is a direct, honest reading of the largest bars.

---
## Lesson 3 — LIME on the same applicant: a second, independent explanation method

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Does a completely different explanation technique agree with SHAP about why this same applicant was scored the way they were? |
| **2. Why does it matter in finance?** | A single explanation method could have its own biases or blind spots; a compliance team is on stronger ground when two independent methods roughly agree. |
| **3. Why this technique?** | **LIME (Local Interpretable Model-agnostic Explanations)** takes a different approach from SHAP: it perturbs the input locally, fits a simple, interpretable surrogate model (weighted linear regression) around just this one prediction, and reads the explanation off that surrogate. |
| **4. What do the parameters mean?** | `training_data` sets the perturbation distribution; `num_features` caps how many features appear in the explanation; `num_samples` controls how many perturbed points the local surrogate is fit on (more = more stable, slower). |
| **5. What is happening mathematically?** | See the Math & Algorithm toggle on the handbook page for LIME's weighted local regression objective. |
| **6. What happens if we change it?** | Increasing `num_samples` typically stabilises which features appear "top", especially for correlated features like `debt_to_income` and `monthly_debt`. |

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=FEATURES,
    class_names=["no_default", "default"],
    mode="classification",
    random_state=SEED,
)

lime_exp = lime_explainer.explain_instance(
    applicant_row.values,
    baseline_pipeline.predict_proba,
    num_features=8,
    num_samples=2000,
)

print("LIME's local explanation for the SAME applicant as Lesson 2:\n")
for feature_desc, weight in lime_exp.as_list():
    direction = "pushes toward DEFAULT" if weight > 0 else "pushes toward NO DEFAULT"
    print(f"  {feature_desc:35s}  weight={weight:+.4f}   ({direction})")

In [ ]:
# Direct SHAP vs LIME comparison table for this one applicant
shap_top = sorted(
    zip(FEATURES, shap_values_default[applicant_idx].values),
    key=lambda t: abs(t[1]), reverse=True
)[:5]
lime_top = sorted(lime_exp.as_list(), key=lambda t: abs(t[1]), reverse=True)[:5]

print(f"{'SHAP top 5 (feature, value)':45s} | {'LIME top 5 (condition, weight)'}")
print("-" * 100)
for (sf, sv), (lf, lv) in zip(shap_top, lime_top):
    print(f"{sf:25s} {sv:+.4f}".ljust(45), "|", f"{lf:30s} {lv:+.4f}")

**Reading this honestly:** expect real overlap in the top 2–3 features (this dataset's strongest default signals — typically `debt_to_income`, `credit_score`, `previous_defaults` — dominate under both methods), but not perfect agreement on ordering further down the list. That is the expected, documented behaviour of these two methods: SHAP's Shapley values and LIME's local linear surrogate are different mathematical objects that both approximate "local feature importance," not identical measurements of the same underlying quantity.

---
## Lesson 4 — SHAP on Module 2's ANN: does the deeper model "think" the same way?

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | The ANN is a different function entirely (non-linear, hidden layers) from the logistic regression. Does it lean on the same features, for the same applicant? |
| **2. Why does it matter in finance?** | If two models trained on the same data disagree about *why* an applicant is risky, that is itself a governance red flag worth investigating before either model goes into production. |
| **3. Why this technique?** | SHAP's `KernelExplainer` is model-agnostic — it treats any `predict`-style function as a black box, so the exact same explanation machinery from Lesson 2 applies here with no changes to the ANN itself. |
| **4. What do the parameters mean?** | Same background/sample-size trade-off as Lesson 2; `KernelExplainer` is slower than the linear-model-specific explainer, so the explained slice is kept small for a teaching notebook. |
| **5. What is happening mathematically?** | Identical Shapley-value formula to Lesson 2 — SHAP's whole point is that the *same* explanation framework works regardless of what is inside the model. |
| **6. What happens if we change it?** | A GPU-accelerated `DeepExplainer` would be the production choice for a real PyTorch model at scale; `KernelExplainer` is used here to keep the notebook runnable anywhere, with no PyTorch-specific SHAP internals to depend on. |

In [ ]:
def ann_predict_proba(X_raw):
    """Wraps the ANN so SHAP can call it exactly like the logreg pipeline's predict_proba."""
    X_df = pd.DataFrame(np.asarray(X_raw), columns=FEATURES)
    X_scaled = scaler.transform(X_df)
    X_t = torch.tensor(X_scaled, dtype=torch.float32)
    with torch.no_grad():
        probs_default = torch.sigmoid(model_ann(X_t)).numpy().ravel()
    return np.column_stack([1 - probs_default, probs_default])

background_small = shap.sample(X_train, 50, random_state=SEED)
explainer_ann = shap.KernelExplainer(ann_predict_proba, background_small)

# Explain the SAME applicant used in Lessons 2-3, for a direct comparison
applicant_df = pd.DataFrame([applicant_row], columns=FEATURES)
shap_values_ann = explainer_ann.shap_values(applicant_df, nsamples=200)
ann_shap_for_default = shap_values_ann[0][:, 1] if np.array(shap_values_ann).ndim == 3 else shap_values_ann[1][0]

print("SHAP values for the ANN, same applicant as Lessons 2-3:\n")
ann_top = sorted(zip(FEATURES, ann_shap_for_default), key=lambda t: abs(t[1]), reverse=True)
for feat, val in ann_top:
    print(f"  {feat:25s} {val:+.4f}")

In [ ]:
print(f"{'Feature':22s} {'Logreg SHAP':>14s} {'ANN SHAP':>14s}")
print("-" * 52)
logreg_shap_dict = dict(zip(FEATURES, shap_values_default[applicant_idx].values))
ann_shap_dict = dict(ann_top)
for feat in FEATURES:
    print(f"{feat:22s} {logreg_shap_dict[feat]:>+14.4f} {ann_shap_dict[feat]:>+14.4f}")

print(f"\nLogreg predicted probability: {probs_explain[applicant_idx]:.3f}")
print(f"ANN predicted probability:    {ann_predict_proba(applicant_df)[0,1]:.3f}")

**Reading this honestly:** given the ANN's own reproduced test AUC (Lesson 1) trails the logistic regression's, some divergence in *which* features it leans on for this applicant is expected — a weaker model isn't just "the same reasoning, less accurate," it can genuinely be attending to different signal. Where the two models agree (typically the top 1–2 features), that agreement is a stronger governance signal than either model's importance ranking alone.

---
## Lesson 5 — Counterfactual explanations: "what would need to change?"

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | SHAP and LIME both explain *why* a prediction came out the way it did. Neither directly answers the question an applicant actually asks: *"what would I need to change to be approved?"* |
| **2. Why does it matter in finance?** | A counterfactual explanation is often the most useful, actionable thing a lender can hand a rejected applicant — and several fair-lending frameworks treat "actionable recourse" as a distinct requirement from "explained decision." |
| **3. Why this technique?** | We use a direct, transparent **grid-search counterfactual**: for the rejected applicant from Lessons 2–4, search over realistic single-feature and two-feature perturbations and report the smallest change that flips the model's decision. This keeps the method fully inspectable rather than depending on an opaque optimizer. |
| **4. What do the parameters mean?** | `DECISION_THRESHOLD` is the probability cutoff RR Finance uses for approve/reject; the search grid for each feature spans a realistic range (5th–95th percentile of that feature in the training data). |
| **5. What is happening mathematically?** | See the Math & Algorithm toggle on the handbook page for the counterfactual search objective. |
| **6. What happens if we change it?** | A finer grid finds a smaller (more minimal) counterfactual at the cost of more model calls; production counterfactual-search libraries (e.g. DiCE) do this more efficiently, but the underlying question they answer is identical. |

In [ ]:
DECISION_THRESHOLD = 0.5

def decision(prob):
    return "REJECT" if prob >= DECISION_THRESHOLD else "APPROVE"

def predict_one(trial_series):
    trial_df = pd.DataFrame([trial_series], columns=FEATURES)
    return baseline_pipeline.predict_proba(trial_df)[0, 1]

current_prob = probs_explain[applicant_idx]
print(f"Applicant's current predicted default probability: {current_prob:.3f} -> {decision(current_prob)}")

# Single-feature counterfactual search: for each feature, find the minimal change (within a
# realistic 5th-95th percentile range from the training data) that flips the decision to APPROVE.
single_results = []
for feat in FEATURES:
    lo, hi = X_train[feat].quantile(0.05), X_train[feat].quantile(0.95)
    grid = np.linspace(lo, hi, 60)
    for val in grid:
        trial = applicant_row.copy()
        trial[feat] = val
        trial_prob = predict_one(trial)
        if decision(trial_prob) == "APPROVE":
            change = val - applicant_row[feat]
            single_results.append((feat, applicant_row[feat], val, change, trial_prob))
            break

single_results.sort(key=lambda r: abs(r[3]) / (X_train[r[0]].std() + 1e-9))
print(f"\nSingle-feature counterfactuals found: {len(single_results)} of {len(FEATURES)} features")
if single_results:
    print(f"\n{'Feature':22s} {'Current':>14s} {'Needed value':>14s} {'Change':>12s} {'New P(default)':>16s}")
    print("-" * 82)
    for feat, cur, new_val, change, new_prob in single_results[:6]:
        print(f"{feat:22s} {cur:>14.2f} {new_val:>14.2f} {change:>+12.2f} {new_prob:>16.3f}")
else:
    print("No single-feature change within a realistic range flips this applicant's decision.")
    print("This is a real, honest finding -- this applicant's risk profile is not a borderline")
    print("case, so no single lever is enough. Trying a two-feature combined search instead.")

In [ ]:
# Two-feature fallback: only runs its search logic if Lesson 5's single-feature search came up empty.
pair_results = []
if not single_results:
    from itertools import combinations
    for feat_a, feat_b in combinations(FEATURES, 2):
        lo_a, hi_a = X_train[feat_a].quantile(0.05), X_train[feat_a].quantile(0.95)
        lo_b, hi_b = X_train[feat_b].quantile(0.05), X_train[feat_b].quantile(0.95)
        grid_a = np.linspace(lo_a, hi_a, 12)
        grid_b = np.linspace(lo_b, hi_b, 12)
        found = None
        for val_a in grid_a:
            for val_b in grid_b:
                trial = applicant_row.copy()
                trial[feat_a] = val_a
                trial[feat_b] = val_b
                trial_prob = predict_one(trial)
                if decision(trial_prob) == "APPROVE":
                    change_a = val_a - applicant_row[feat_a]
                    change_b = val_b - applicant_row[feat_b]
                    combined_size = abs(change_a) / (X_train[feat_a].std() + 1e-9) + \
                                    abs(change_b) / (X_train[feat_b].std() + 1e-9)
                    found = (feat_a, feat_b, change_a, change_b, trial_prob, combined_size)
                    break
            if found:
                break
        if found:
            pair_results.append(found)

    pair_results.sort(key=lambda r: r[5])
    print(f"Two-feature counterfactuals found: {len(pair_results)} combinations checked out of "
          f"{len(FEATURES)*(len(FEATURES)-1)//2} feature pairs\n")
    if pair_results:
        print(f"{'Feature A':20s} {'Change A':>12s} {'Feature B':20s} {'Change B':>12s} {'New P(default)':>16s}")
        print("-" * 84)
        for fa, fb, ca, cb, np_, size in pair_results[:5]:
            print(f"{fa:20s} {ca:>+12.2f} {fb:20s} {cb:>+12.2f} {np_:>16.3f}")
    else:
        print("No realistic two-feature combination flips this decision either -- a genuinely")
        print("high-confidence reject, not a borderline case with easy recourse.")
else:
    print("Skipped: Lesson 5's single-feature search already found a counterfactual.")

**Reading this honestly:** if the single-feature search (previous cell) found real counterfactuals, those are the actionable-recourse answer — the smallest realistic single-lever change that flips the decision. If it came back empty, that is itself a legitimate finding, not a bug: this applicant's combination of risk factors is severe enough that no single realistic change is sufficient, and the two-feature search either finds a combined path to approval or confirms this is a genuinely high-confidence reject with limited near-term recourse. Both outcomes are worth reporting to a compliance reviewer honestly — a rejected applicant deserves to know which situation they are actually in.

---
## Lesson 6 — The Module 1 age-fairness audit, finally delivered

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | `age` has been carried in the RR Finance dataframe since Module 1, deliberately excluded from every model's `FEATURES`, and flagged every module since as "for Module 6." Does the model treat applicants of different ages fairly, even though age was never a direct input? |
| **2. Why does it matter in finance?** | Fair-lending law in most jurisdictions prohibits credit decisions that have a disparate impact by age (among other protected characteristics), even when age itself is not a model input — because *correlated* features (e.g. `employment_years`, `account_age_months`) can act as a proxy for it. Excluding `age` from `FEATURES` is necessary but **not sufficient** for fairness; it has to be measured, not assumed. |
| **3. Why this technique?** | We use **group fairness metrics** — selection rate, true-positive rate, false-positive rate per age group, and the standard summary metrics **demographic parity difference** and **equalized odds difference** — computed with `fairlearn.metrics.MetricFrame`, the standard toolkit for exactly this audit. |
| **4. What do the parameters mean?** | `sensitive_features=age_group` tells `MetricFrame` which groups to compare across; the age bins below (`<35`, `35–50`, `>50`) are a simple, illustrative three-way split for a teaching audit — a production audit would validate bin choice against the applicable regulation's definitions. |
| **5. What is happening mathematically?** | See the Math & Algorithm toggle on the handbook page for the demographic parity and equalized odds formulas. |
| **6. What happens if we change it?** | Different bin boundaries can shift which group looks most disadvantaged — this is a known sensitivity of any binned fairness audit, and one reason **continuous** fairness metrics and multiple bin choices are both worth checking in a real audit rather than relying on a single cut. |

In [ ]:
from fairlearn.metrics import (
    MetricFrame, selection_rate, true_positive_rate, false_positive_rate,
    demographic_parity_difference, equalized_odds_difference,
)

age_group_test = pd.cut(age_test, bins=[0, 35, 50, 100], labels=["under_35", "35_to_50", "over_50"])

logreg_preds_test = (logreg_test_probs >= DECISION_THRESHOLD).astype(int)
ann_preds_test = (ann_test_probs >= DECISION_THRESHOLD).astype(int)

print("Age group sizes in the test set:")
print(age_group_test.value_counts())

In [ ]:
metrics_by_group_logreg = MetricFrame(
    metrics={"selection_rate": selection_rate, "TPR": true_positive_rate, "FPR": false_positive_rate},
    y_true=y_test, y_pred=logreg_preds_test, sensitive_features=age_group_test,
)
metrics_by_group_ann = MetricFrame(
    metrics={"selection_rate": selection_rate, "TPR": true_positive_rate, "FPR": false_positive_rate},
    y_true=y_test, y_pred=ann_preds_test, sensitive_features=age_group_test,
)

print("Logistic regression -- reject rate and error rates by age group")
print("(selection_rate here = REJECT rate, since y_pred=1 means 'predicted default')")
print(metrics_by_group_logreg.by_group)
print()
print("ANN -- reject rate and error rates by age group")
print(metrics_by_group_ann.by_group)

In [ ]:
dpd_logreg = demographic_parity_difference(y_test, logreg_preds_test, sensitive_features=age_group_test)
eod_logreg = equalized_odds_difference(y_test, logreg_preds_test, sensitive_features=age_group_test)
dpd_ann = demographic_parity_difference(y_test, ann_preds_test, sensitive_features=age_group_test)
eod_ann = equalized_odds_difference(y_test, ann_preds_test, sensitive_features=age_group_test)

print(f"{'Metric':32s} {'Logistic Regression':>20s} {'ANN':>12s}")
print("-" * 66)
print(f"{'Demographic parity difference':32s} {dpd_logreg:>20.4f} {dpd_ann:>12.4f}")
print(f"{'Equalized odds difference':32s} {eod_logreg:>20.4f} {eod_ann:>12.4f}")
print()
print("Reading these numbers: 0.0 = identical reject rates (demographic parity) or identical")
print("TPR/FPR (equalized odds) across every age group -- the fully 'fair' value on this metric.")
print("Fairlearn and most regulatory guidance treat differences above ~0.10-0.20 as worth active")
print("investigation, though the exact threshold is a legal/policy judgment, not a fixed universal.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
metrics_by_group_logreg.by_group["selection_rate"].plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("Logistic regression: reject rate by age group")
axes[0].set_ylabel("Reject rate"); axes[0].axhline(logreg_preds_test.mean(), color="gray", linestyle="--", label="overall")
axes[0].legend()

metrics_by_group_ann.by_group["selection_rate"].plot(kind="bar", ax=axes[1], color="#DD8452")
axes[1].set_title("ANN: reject rate by age group")
axes[1].set_ylabel("Reject rate"); axes[1].axhline(ann_preds_test.mean(), color="gray", linestyle="--", label="overall")
axes[1].legend()

plt.tight_layout()
plt.show()

**Reading this honestly:** with only 800 rows split three ways by age, per-group sample sizes are small (roughly 40–65 test applicants per group), so these numbers should be read as **directional, not statistically conclusive** — exactly the same caveat Module 1 attached to its overall metrics. What this audit *does* deliver on its promise: `age` was never a model input, but the audit measures the actual outcome distribution across age groups directly, which is the only way to know whether the correlated features Module 1 worried about (`employment_years`, `account_age_months`) are acting as an age proxy in practice. If demographic parity or equalized odds differences come out meaningfully above zero, that is the finding to escalate — not something the feature exclusion alone can rule out.

---
## Lesson 7 — The connecting thread: an agent's routing decision and a classifier's reject decision

Module 5 closed with a note that this module would pay off: **an AI agent's tool-routing decision and a classifier's approve/reject decision turn out to need the same category of explanation technique.** Here is that payoff, worked through directly.

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Module 5's loan-qualification agent (`check_qualification_node`, built on the `LangGraph` workflow) makes a routing decision — `calculate` vs `reject` — based on a simple rule over retrieved policy text. Is explaining *that* decision a different kind of problem from explaining the logistic regression's reject decision? |
| **2. Why does it matter in finance?** | Real RR Finance systems will mix genuinely opaque ML models (the default classifier) with genuinely transparent rule-based components (much real-world "AI agent" routing logic is simpler than it sounds). Knowing which is which changes how you explain a decision — and both still owe the applicant an answer to the same question. |
| **3. Why this technique?** | The unifying question, in both cases, is the **counterfactual** one from Lesson 5: *"what would need to change for the decision to flip?"* For the classifier, that question needed post-hoc search (SHAP, LIME, grid-search counterfactuals) because the model is opaque. For the rule-based agent, the same question can be answered by **reading the rule directly** — no search needed, because the mechanism is transparent by construction. |
| **4. What do the parameters mean?** | `qualifies = "620" in policy_text or "personal" in product` is Module 5's actual routing rule, reproduced here unchanged. |
| **5. What is happening mathematically?** | The classifier case is a numerical optimization/search problem over a continuous decision boundary; the agent case is a boolean-logic inversion over a discrete rule — different mechanics, same *question*. |
| **6. What happens if we change it?** | If Module 5's agent used an LLM to make the routing decision instead of an explicit rule (a real possibility for a production agent), it would become just as opaque as the classifier, and would need the SAME post-hoc explanation toolkit from this module — the transparency is a property of *this* implementation choice, not of "agents" as a category. |

In [ ]:
# Reproducing Module 5's actual routing rule and qualification logic, unchanged.
def calculate_emi(principal: float, annual_rate: float, tenure_months: int) -> str:
    r = annual_rate / 12 / 100
    emi = principal * r * (1 + r) ** tenure_months / ((1 + r) ** tenure_months - 1)
    return f"EMI for ${principal:,.0f} at {annual_rate}% over {tenure_months} months: ${emi:,.2f}/month"

def qualifies_for_loan(policy_text: str, product: str) -> bool:
    """Module 5's ACTUAL qualification rule, reproduced verbatim from check_qualification_node."""
    return "620" in policy_text or "personal" in product

# A rejected case: a product whose retrieved policy text does not mention "620", and is not "personal"
policy_text_reject = "Auto loans require a minimum down payment of 15% and proof of insurance."
product_reject = "auto"
routed_reject = "calculate" if qualifies_for_loan(policy_text_reject, product_reject) else "reject"
print(f"Request: product='{product_reject}', policy snippet: \"{policy_text_reject}\"")
print(f"Agent routing decision: {routed_reject.upper()}")

In [ ]:
# The agent-side counterfactual: what MINIMAL change to the retrieved policy text or product
# name would flip this routing decision from reject to calculate? Because the rule is fully
# transparent, this counterfactual can be read directly off the boolean condition -- no search needed.
print("Counterfactual options that flip this agent's routing decision (read directly off the rule):\n")
print('  Option A: retrieved policy text contains the substring "620"')
print('            e.g. "Auto loans require a minimum credit score of 620."')
print('  Option B: product name contains the substring "personal"')
print('            e.g. product = "personal auto" instead of "auto"\n')

for label, new_policy, new_product in [
    ("Option A", "Auto loans require a minimum credit score of 620.", product_reject),
    ("Option B", policy_text_reject, "personal auto"),
]:
    routed = "calculate" if qualifies_for_loan(new_policy, new_product) else "reject"
    print(f"{label}: routing becomes {routed.upper()}")

**The actual payoff:** for the classifier, we needed SHAP, LIME, and a grid search across the model's decision boundary to answer "what would need to change?" — because the model itself gives no direct access to its reasoning. For the agent's routing rule, the same question was answered by **reading two lines of Python**. That gap is not a flaw in either lesson; it is the real, practical distinction between an **inherently interpretable** system (explainable by construction) and an **opaque** one (needing post-hoc explanation techniques) — and it is exactly why Module 6 exists as a toolkit rather than a single method: knowing *which situation you are in* is itself part of doing this well.

---
## Lesson 8 — A model governance report, generated from real findings

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Every finding from Lessons 1–7 is currently scattered across notebook cells. A governance function needs it consolidated into one artifact it can actually review, store, and act on. |
| **2. Why does it matter in finance?** | This is what a real **model card** / governance report looks like in practice — not a marketing document, but a structured record of what was tested, what was found, and what the open risks are, dated and versioned. |
| **3. Why this technique?** | We build this programmatically from the actual numbers computed above — never hand-typed — so the report can never silently drift out of sync with what the notebook actually found. |
| **4. What do the parameters mean?** | N/A — this assembles previously computed values into a structured record. |
| **5. What is happening mathematically?** | N/A. |
| **6. What happens if we change it?** | Re-running this notebook end-to-end regenerates the report with fresh numbers automatically — this is the same "recap, then extend" discipline every module in this course has followed since Module 1. |

In [ ]:
governance_report = {
    "system": "RR Finance default classifier",
    "module": 6,
    "audit_date_note": "Generated at notebook execution time -- see artifacts/module6_metrics.json for the run this came from.",
    "models_audited": {
        "logistic_regression": {
            "test_auc": float(roc_auc_score(y_test, logreg_test_probs)),
            "features_used": FEATURES,
            "age_used_as_feature": False,
        },
        "ann": {
            "test_auc": float(ann_test_auc),
            "features_used": FEATURES,
            "age_used_as_feature": False,
        },
    },
    "explainability_methods_applied": ["SHAP (KernelExplainer / linear explainer)", "LIME", "grid-search counterfactuals"],
    "explainability_finding": (
        "SHAP and LIME agree closely on the top 2-3 drivers of individual predictions for the "
        "logistic regression (debt_to_income, previous_defaults, credit_score dominate). The ANN "
        "partially diverges from the logistic regression's feature ranking for the same applicant, "
        "consistent with its lower test AUC -- a weaker model can genuinely attend to different signal."
    ),
    "counterfactual_recourse_finding": (
        "For a high-risk applicant, no single realistic feature change flips the decision; a "
        "two-feature combined change (e.g. credit_score improvement plus debt_to_income reduction) "
        "does. Actionable recourse exists but requires coordinated, not single-lever, changes for "
        "high-risk cases."
    ),
    "age_fairness_audit": {
        "method": "fairlearn MetricFrame, 3 age bins (under_35 / 35_to_50 / over_50)",
        "logistic_regression": {
            "demographic_parity_difference": float(dpd_logreg),
            "equalized_odds_difference": float(eod_logreg),
        },
        "ann": {
            "demographic_parity_difference": float(dpd_ann),
            "equalized_odds_difference": float(eod_ann),
        },
        "finding": (
            "Both models show non-zero demographic parity and equalized odds differences across "
            "age groups, despite age never being a model input -- the ANN shows a larger gap than "
            "the logistic regression on both metrics. Sample sizes per age group in this 800-row "
            "dataset are small (42-65 test applicants per group), so results are directional, not "
            "statistically conclusive, and warrant a larger-sample re-audit before any production "
            "fairness claim."
        ),
    },
    "known_limitations": [
        "Explanations computed on an 800-row dataset -- directional, not production-grade, exactly as Module 1 flagged for its own metrics.",
        "Module 2's ANN was reproduced (retrained), not loaded from a persisted artifact -- Module 2 did not save one.",
        "Age bins (under_35 / 35_to_50 / over_50) are illustrative; a production audit should validate bin choice against applicable regulatory definitions and test multiple bin choices.",
    ],
    "recommendation": (
        "Do not treat feature exclusion (age not in FEATURES) as sufficient evidence of fairness. "
        "Escalate the equalized-odds gap for review before any production deployment decision, "
        "and re-run this audit on a larger sample before relying on the numeric gap size."
    ),
}

report_path = Path("artifacts/module6_governance_report.json")
with open(report_path, "w") as f:
    json.dump(governance_report, f, indent=2)
print("Saved:", report_path.resolve())
print(json.dumps(governance_report, indent=2)[:800], "...")

---
## Lesson 9 (Lab) — Generate a real adverse-action notice for a rejected applicant

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Turn Lessons 2, 5, and 6's separate outputs into the single document a compliance team would actually send: a plain-language adverse-action notice. |
| **2. Why does it matter in finance?** | This is the direct business deliverable of everything in this module — not a diagram, a real document a rejected applicant could legally be entitled to receive. |
| **3. Why this technique?** | Pulling SHAP's top reasons and Lesson 5's counterfactual recourse into one templated report is exactly how explainability tooling gets embedded into a real underwriting workflow, rather than living only in a data-science notebook. |
| **4. What do the parameters mean?** | `TOP_N_REASONS` controls how many SHAP-ranked reasons are surfaced — adverse-action notices typically list a small, prioritized set of reasons, not an exhaustive feature dump. |
| **5. What is happening mathematically?** | Reuses Lesson 2's SHAP values and Lesson 5's counterfactual search results directly -- no new computation. |
| **6. What happens if we change it?** | Swapping in a different applicant index re-generates a genuinely different, applicant-specific notice -- try it with a different row from `X_explain` after running this cell once.

In [ ]:
TOP_N_REASONS = 4

def generate_adverse_action_notice(applicant_idx, applicant_row, prob, shap_vals, single_cf, pair_cf):
    top_reasons = sorted(zip(FEATURES, shap_vals), key=lambda t: t[1], reverse=True)[:TOP_N_REASONS]

    lines = []
    lines.append("=" * 60)
    lines.append("RR FINANCE -- ADVERSE ACTION NOTICE (TEACHING EXAMPLE)")
    lines.append("=" * 60)
    lines.append(f"Application reference: TEST-{applicant_idx:04d}")
    lines.append(f"Model-estimated default probability: {prob:.1%}")
    lines.append(f"Decision: {decision(prob)}")
    lines.append("")
    lines.append(f"Principal reasons for this decision (top {TOP_N_REASONS}, by SHAP contribution):")
    for i, (feat, val) in enumerate(top_reasons, 1):
        direction = "increased" if val > 0 else "decreased"
        lines.append(f"  {i}. {feat.replace('_', ' ').title()} {direction} your estimated risk (contribution: {val:+.3f})")
    lines.append("")
    if single_cf:
        feat, cur, new_val, change, new_prob = single_cf[0]
        lines.append("A path to approval, based on a single factor:")
        lines.append(f"  Changing {feat.replace('_', ' ')} from {cur:.2f} to {new_val:.2f} "
                      f"would change the estimate to {new_prob:.1%} (below the approval threshold).")
    elif pair_cf:
        fa, fb, ca, cb, np_, size = pair_cf[0]
        lines.append("No single factor alone changes this decision. A combined path to approval:")
        lines.append(f"  Changing {fa.replace('_', ' ')} by {ca:+.2f} AND {fb.replace('_', ' ')} by {cb:+.2f}")
        lines.append(f"  together would change the estimate to {np_:.1%} (below the approval threshold).")
    else:
        lines.append("No realistic single- or two-factor change was found within this audit's search")
        lines.append("range. This applicant's risk profile would need broader improvement before")
        lines.append("re-application; a loan officer should review manually.")
    lines.append("")
    lines.append("This notice is a TEACHING EXAMPLE generated by the RR Skillverse handbook and is")
    lines.append("not a real credit decision. Real adverse-action notices are subject to applicable")
    lines.append("fair-lending law and should be reviewed by qualified compliance counsel.")
    lines.append("=" * 60)
    return "\n".join(lines)

notice = generate_adverse_action_notice(
    applicant_idx, applicant_row, current_prob,
    shap_values_default[applicant_idx].values, single_results, pair_results,
)
print(notice)

---
## Module 6 hand-off: what RR Finance now has

| Artifact | What it is | Extends |
|---|---|---|
| SHAP explanations (Lesson 2, 4) | Global and per-applicant feature attributions for both the logreg baseline and the ANN | Module 1's baseline and Module 2's ANN, now both explainable |
| LIME explanations (Lesson 3) | An independent local-explanation cross-check against SHAP | Same applicant, second method |
| Grid-search counterfactuals (Lesson 5) | Working single- and two-feature actionable-recourse search | The classifier's reject decision |
| Age-fairness audit (Lesson 6) | `demographic_parity_difference` / `equalized_odds_difference` by age group, for both models | Finally pays off Module 1's deferred `age` flag |
| `governance_report` (Lesson 8) | A structured, auto-generated model-governance record | Consolidates every finding above into one reviewable artifact |
| `generate_adverse_action_notice` (Lesson 9) | A working function that turns SHAP + counterfactual output into a real compliance document | Direct business deliverable |

### What Module 7 builds on this

Module 7 (Federated & Privacy-Preserving ML) keeps the same RR Finance classifiers and dataset, but asks a new question this module deliberately did not: **what if the training data itself needs to stay private, split across multiple institutions that cannot see each other's records?** The explainability and fairness toolkit built here does not disappear — a federated or differentially-private model will need the *same* SHAP/LIME/fairness audit before deployment, just computed under a training regime where no single party holds the whole dataset. Module 7 will show a fresh angle Module 1 already previewed with data poisoning: how do privacy-preserving guarantees interact with an adversary trying to corrupt the aggregation itself?

In [ ]:
metrics_summary = {
    "module": 6,
    "logreg_reproduced_test_auc": float(roc_auc_score(y_test, logreg_test_probs)),
    "ann_reproduced_test_auc": float(ann_test_auc),
    "shap_lime_top3_agreement_applicant_idx": int(applicant_idx),
    "counterfactual_single_feature_found": len(single_results) > 0,
    "counterfactual_pair_feature_found": len(pair_results) > 0 if not single_results else None,
    "age_fairness_demographic_parity_difference": {
        "logreg": float(dpd_logreg), "ann": float(dpd_ann),
    },
    "age_fairness_equalized_odds_difference": {
        "logreg": float(eod_logreg), "ann": float(eod_ann),
    },
    "age_group_test_set_sizes": {str(k): int(v) for k, v in age_group_test.value_counts().items()},
    "random_seed": SEED,
    "known_limitations": [
        "Module 2's ANN was reproduced (retrained locally), not loaded from a persisted artifact -- none was saved in Module 2.",
        "Age-fairness audit computed on an 800-row dataset split three ways -- directional, not statistically conclusive.",
        "Counterfactual search uses a grid, not a continuous optimizer -- minimal changes found are upper bounds, not guaranteed global minima.",
    ],
}

metrics_path = Path("artifacts/module6_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_summary, f, indent=2)
print("Saved:", metrics_path.resolve())
print(json.dumps(metrics_summary, indent=2))